In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
     Implement a quantized matrix multiplication program for 8-bit signed integer matrices. Given two input matrices <code>A</code> of dimensions $M \times K$ and <code>B</code> of dimensions $K \times N$, quantization scales <code>scale_A</code>, <code>scale_B</code>, output scale <code>scale_C</code>, zero-points <code>zero_point_A</code>, <code>zero_point_B</code>, <code>zero_point_C</code>, compute:
     $$
     C_{\text{quant}}(i, j) = \mathrm{clamp}\left(
         \mathrm{round}\left(
             \frac{
                 \sum_{k=0}^{K-1} (A_{ik} - z_A)(B_{kj} - z_B) \cdot s_A s_B
             }{s_C}
         \right) + z_C,\ -128,\ 127
     \right)
     $$
     where <code>s_A = scale_A</code>, <code>z_A = zero_point_A</code>, etc.
     </p>

     <h2>Implementation Requirements</h2>
     <ul>
     <li>External libraries are not permitted</li>
     <li>The <code>solve</code> function signature must remain unchanged</li>
     <li>The final result must be stored in the output matrix <code>C</code> as <code>int8</code></li>
     <li>After accumulation in int32 and scaling in float32, values must be rounded to the nearest integer, shifted by <code>zero_point_C</code>, and clamped to the <code>[-128, 127]</code> range</li>
     </ul>

     <h2>Example 1:</h2>
     <pre>
     Input:
     A = [[1, 2],
          [3, 4]]
     B = [[5, 6],
          [7, 8]]
     M = 2, N = 2, K = 2
     scale_A = 0.1, scale_B = 0.2, scale_C = 0.05
     zero_point_A = 0, zero_point_B = 0, zero_point_C = 0

     Output:
     C = [[19, 22],
          [43, 50]]
     </pre>

     <h2>Example 2:</h2>
     <pre>
     Input:
     A = [[1, 2]]
     B = [[3],
          [4]]
     M = 1, N = 1, K = 2
     scale_A = 1.0, scale_B = 1.0, scale_C = 1.0
     zero_point_A = 1, zero_point_B = 3, zero_point_C = 5

     Output:
     C = [[6]]
     </pre>

     <h2>Constraints</h2>
     <ul>
     <li>1 ≤ <code>M</code>, <code>N</code>, <code>K</code> ≤ 4096</li>
     <li><code>scale_A</code>, <code>scale_B</code>, <code>scale_C</code> are positive floats</li>
     <li><code>-128</code> ≤ <code>zero_point_A</code>, <code>zero_point_B</code>, <code>zero_point_C</code> ≤ <code>127</code></li>

  <li>Performance is measured with <code>K</code> = 2,048, <code>M</code> = 8,192, <code>N</code> = 4,096</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// A, B, C are device pointers
extern "C" void solve(const int8_t* A, const int8_t* B, int8_t* C, int M, int N, int K,
                      float scale_A, float scale_B, float scale_C, int zero_point_A,
                      int zero_point_B, int zero_point_C) {}


# CUTE

In [ ]:
%%writefile solution_cute.py
import cutlass
import cutlass.cute as cute


# A, B, C are tensors on the GPU
@cute.jit
def solve(
    A: cute.Tensor,
    B: cute.Tensor,
    C: cute.Tensor,
    M: cute.Int32,
    N: cute.Int32,
    K: cute.Int32,
    scale_A: cute.Float32,
    scale_B: cute.Float32,
    scale_C: cute.Float32,
    zero_point_A: cute.Int32,
    zero_point_B: cute.Int32,
    zero_point_C: cute.Int32,
):
    pass


# JAX

In [ ]:
%%writefile solution_jax.py
import jax
import jax.numpy as jnp


# A, B are tensors on the GPU
@jax.jit
def solve(
    A: jax.Array,
    B: jax.Array,
    M: int,
    N: int,
    K: int,
    scale_A: float,
    scale_B: float,
    scale_C: float,
    zero_point_A: int,
    zero_point_B: int,
    zero_point_C: int,
) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


@export
def solve(
    A: UnsafePointer[Int8, MutExternalOrigin],
    B: UnsafePointer[Int8, MutExternalOrigin],
    C: UnsafePointer[Int8, MutExternalOrigin],
    M: Int32,
    N: Int32,
    K: Int32,
    scale_A: Float32,
    scale_B: Float32,
    scale_C: Float32,
    zero_point_A: Int32,
    zero_point_B: Int32,
    zero_point_C: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution_pytorch.py
import torch


# A, B, C are tensors on the GPU
def solve(
    A: torch.Tensor,
    B: torch.Tensor,
    C: torch.Tensor,
    M: int,
    N: int,
    K: int,
    scale_A: float,
    scale_B: float,
    scale_C: float,
    zero_point_A: int,
    zero_point_B: int,
    zero_point_C: int,
):
    pass


# Triton

In [ ]:
%%writefile solution_triton.py
import torch
import triton
import triton.language as tl


# a, b, c are tensors on the GPU
def solve(
    a: torch.Tensor,
    b: torch.Tensor,
    c: torch.Tensor,
    M: int,
    N: int,
    K: int,
    scale_A: float,
    scale_B: float,
    scale_C: float,
    zero_point_A: int,
    zero_point_B: int,
    zero_point_C: int,
):
    pass


# Evaluate Setup

In [ ]:
# Download required files from GitHub
!mkdir -p core
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/challenge_base.py -O core/challenge_base.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/evaluator.py -O core/evaluator.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/medium/32_int8_quantized_matmul/challenge.py -O challenge.py

from challenge import Challenge
from core.evaluator import Evaluate

ch = Challenge()


# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch, EVAL_LANG)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
